In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'Where do I watch the live office hours or workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'Where do I watch the live office hours or workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [9]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [10]:
q['question']

'Where do I watch the live office hours or workshop stream if I’m a student?'

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

0.00150225

In [13]:
print(answer)

Students watch the live office hours/workshop stream on **YouTube Live** via the **DataTalksClub YouTube channel**. The video URL is also posted in the **announcements channel on Telegram and Slack** before it starts, and the **Slido link** is pinned in the chat when live.


In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Where do I watch the live office hours or workshop stream if I’m a student?',
 'answer_llm': 'Students watch the live office hours/workshop stream on **YouTube Live** via the **DataTalksClub YouTube channel**. The video URL is also posted in the **announcements channel on Telegram and Slack** before it starts, and the **Slido link** is pinned in the chat when live.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
record = generate_rag_answer(q)
record

{'question': 'Where do I watch the live office hours or workshop stream if I’m a student?',
 'answer_llm': 'Students watch the live office hours/workshop stream on **YouTube Live**. The video link is posted in the **announcements channel on Telegram and Slack** before it begins, and you can also watch it on the **DataTalksClub YouTube channel**.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [18]:
assistant.total_cost()

0.0024112499999999998

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
print("Anzahl Ground-Truth-Einträge:", len(ground_truth))
print("Anzahl Dokumente in doc_idx:", len(doc_idx))

missing_ids = sorted({
    rec["document"]
    for rec in ground_truth
    if rec["document"] not in doc_idx
})

print("Anzahl fehlender IDs:", len(missing_ids))
print("Fehlende IDs:", missing_ids)

Anzahl Ground-Truth-Einträge: 590
Anzahl Dokumente in doc_idx: 118
Anzahl fehlender IDs: 0
Fehlende IDs: []


In [23]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [24]:
results[:10]

[{'question': 'I just found this course, is it too late to join?',
  'answer_llm': 'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still sign up even if the course already started?',
  'answer_llm': "I don't know.",
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still get a certificate somehow?',
  'answer_llm': 'Yes — if you join now, you can still get a certificate **only if** you submit your project while a **live cohort is still accepting submissions**. You can work through the course in self-paced mode, but the **certificate requires**:\n\n- finishing the *

In [25]:
df_results = pd.DataFrame(results)

In [26]:
df_results.head()

,question,answer_llm,answer_orig,document
0,"I just found this course, is it too late to join?","Yes, you can still join. If you want a certifi...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still sign up even if the course already...,I don't know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, can I still get a certificate s...","Yes — if you join now, you can still get a cer...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the ce...,"If you’re late to the course, you can still ge...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Are late joiners allowed, and does that affect...","Yes, late joiners can catch up, but the certif...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [27]:
assistant.total_cost()


0.6405360000000004

In [28]:
df_results.to_csv("data/rag-answers-new.csv", index=False)